# Datamine DISCLA Process Example

This notebook demonstrates how to configure and run the **`discla`** process wrapper in `dmstudio`.

## Process Description

## DISCLA

Multiple discriminant classification of unknown samples into groups using the discriminant functions and group centroids calculated in the [DISCAN](<discan.md>) process.

Discriminant analysis is a classification problem, where two or more groups or clusters or populations are known a priori and one or more new observations are classified into one of the known populations based on the measured characteristics.

Multiple discriminant analysis using trial (test) data sets to calculate scores and discriminant functions with group centroids for use in the **DISCLA** process.

The calculation of discriminant functions (Discriminant analysis) will establish a set of rules using geological control groups or training sets which can then be applied to the classification of unknown samples using the **DISCLA** process.

For example, test data from DISCAN will calculate discriminant functions and group centroids to classify known samples into mineralized and non-mineralized. DISCLA will then use the discriminant functions and group centroids and apply them to unknown samples in order to identified them into the classes mineralized and non-mineralized, plus a third class, to include the samples which are not classifiable.

**Note** : there is a maximum limit of 45 variables and 10 classifier groups with an extra group for non classifiable samples. Samples containing absent data are ignored.

### File Handling

The input file (&**IN**), of samples for classification must contain a sample identifier field (@**SAMPID**). The input files (&**INFUNCTS**) discriminant functions and (&**INCENTRD**) group centroids are obligatory. File &**INCENTRD** must contain a group identifier specified in @**GROUPID**. The output file of classified samples (&**OUT**) contains a new field named in @**GROUPID** identifying the group classifier or a flag to note that the sample is not classifiable using the data given in files &**INFUNCTS** and &**INCENTRD**.

The output file containing the discriminant scores for each sample can be joined to the original data containing the grid coordinates using the **SAMPID** as a key field. The scores can then be plotted as a map.

### Input Files:

* **in** (Undefined):
  Input file. Must contain sample identity field.
  Required=Yes

* **infuncts** (Undefined):
  Input file containing discriminant functions. Produced from process **DISCAN**.
  Required=Yes

* **incentrd** (Undefined):
  Input file containing group centroids. Produced from process **DISCAN**. Must contain a
  group identity field specified in **GROUPID**.
  Required=Yes

### Output Files:

* **out** (Undefined):
  Output file containing samples classified into groups identified by the GROUPID field
  and the discriminant scores..
  Required=Yes

### Fields:

* **groupid** (Alphanumeric : INCENTRD):
  Compulsory group identifier field contained in **INCENTRD**.
  Default=Undefined
  Required=Yes

* **sampid** (Any : IN):
  Compulsory sample identifier field in input file **IN**.
  Default=Undefined
  Required=Yes

* **fields** (Undefined : Undefined):
  First field to be used. No fields specified means all.
  Default=Undefined
  Required=No

### Parameters:

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('discla')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically (4 levels up from subfolders)
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute discla
print("Running discla...")
dm_cmd.discla(
    in_i='t_assays',  # required
    infuncts_i=['t_input_file'],  # required
    incentrd_i='t_input_file',  # required
    out_o='t_discla_out',  # required
    groupid_f='optional',  # required
    sampid_f='optional',  # required
    fields_f=['AU'],  # required
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("discla execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_discla_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")